# P2.4 · Analiza danych

Profile dobowe i sezonowe zapotrzebowania netto, krzywa kaczki, błąd prognozy PSE z planu dobowego.

Logika: `p2/analysis.py`, `p2/plots.py`, `p2/metrics.py`. Dane: widoki godzinowe z bazy (P2.3) — przed uruchomieniem `docker compose up -d` i `uv run kse-data load`. Opis wyników: `docs/etapy/P2.4.md`.

In [1]:
from common import db
from common.config import PROJECT_ROOT, load_settings
from p2 import analysis, plots

FIG = PROJECT_ROOT / 'docs' / 'fig'
df = analysis.load_hourly(db.make_engine(load_settings().database_url))
print(f"godzin: {len(df)}, doby: {df['business_date'].min()} – {df['business_date'].max()}")

godzin: 21180, doby: 2024-06-14 – 2026-11-13


In [2]:
tables = plots.render_all(df, FIG)

## Profile dobowe wg pory roku

![p2_4_profil_sezonowy_netto.png](../../docs/fig/p2_4_profil_sezonowy_netto.png)

In [3]:
tables['profil_sezon'].round(0)

season,zima,wiosna,lato,jesień
hour_local,,,,
0,16938.0,15228.0,14739.0,15413.0
1,16415.0,14705.0,14121.0,14884.0
2,16134.0,14457.0,13788.0,14596.0
3,16129.0,14473.0,13692.0,14613.0
4,16454.0,14701.0,13615.0,14930.0
5,17304.0,15053.0,13630.0,15766.0
6,19371.0,16010.0,14360.0,17668.0
7,20456.0,15072.0,13728.0,18164.0
8,20598.0,12981.0,12141.0,17173.0


## Krzywa kaczki

![p2_4_krzywa_kaczki_czerwiec.png](../../docs/fig/p2_4_krzywa_kaczki_czerwiec.png)

![p2_4_skladowe_czerwiec.png](../../docs/fig/p2_4_skladowe_czerwiec.png)

![p2_4_glebokosc_kaczki.png](../../docs/fig/p2_4_glebokosc_kaczki.png)

In [4]:
tables['glebokosc_kaczki'].round(0)

,midday_min,evening_max,depth
month,,,
2024-06,10183.0,18984.0,8801.0
2024-07,10235.0,18843.0,8608.0
2024-08,9470.0,18973.0,9504.0
2024-09,10522.0,19759.0,9237.0
2024-10,13291.0,20999.0,7709.0
2024-11,18182.0,21731.0,3549.0
2024-12,18781.0,21360.0,2579.0
2025-01,18590.0,22302.0,3712.0
2025-02,16223.0,22713.0,6491.0


## Średnie miesięczne

![p2_4_srednie_miesieczne.png](../../docs/fig/p2_4_srednie_miesieczne.png)

In [5]:
tables['miesiace'].round(0)

,demand,pv,net_demand
month,,,
2024-06,17758.0,3497.0,14261.0
2024-07,17501.0,3328.0,14173.0
2024-08,17063.0,3196.0,13867.0
2024-09,17195.0,2438.0,14757.0
2024-10,18329.0,1623.0,16706.0
2024-11,19255.0,603.0,18652.0
2024-12,18991.0,380.0,18611.0
2025-01,19783.0,579.0,19205.0
2025-02,20295.0,1396.0,18899.0


## Błąd prognozy PSE (prognoza − wykonanie)

![p2_4_blad_pse_godziny.png](../../docs/fig/p2_4_blad_pse_godziny.png)

![p2_4_blad_pse_miesiace.png](../../docs/fig/p2_4_blad_pse_miesiace.png)

In [6]:
display(tables['pse_metryki'].round(2))

,MAE,RMSE,MAPE,bias,n
netto,521.75,770.86,4.10,-30.93,21180.0
zapotrzebowanie,372.13,497.11,2.07,101.26,21180.0
PV,316.46,621.86,NaN,132.19,21180.0
netto — szczyt dobowy,324.96,NaN,NaN,80.82,883.0
netto — minimum południowe,944.84,NaN,NaN,218.42,883.0
